In [2]:
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

load_dotenv()

# 1. Initialize the model
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
    max_tokens=500,
)


In [3]:
class BlogState(TypedDict):

    title: str
    outline : str
    content : str

In [4]:
def create_outline(state:BlogState)->BlogState:

    #fetch title
    title=state['title']

    #call llm gen outline
    prompt=f'Generate a detail outline for a blog on the topic - {title}'
    outline=model.invoke(prompt).content

    #update state
    state['outline']=outline

    return state

In [12]:
def create_blog(state:BlogState)->BlogState:

    title=state['title']
    outline=state['outline']

    prompt=f'Write a detailed blog on the title-{title} using the following outline \n {outline} in 500 word'

    content=model.invoke(prompt).content

    state['content']=content

    return state

In [13]:
graph=StateGraph(BlogState)

#nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

#edges

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

workflow=graph.compile()

In [14]:
initial_state={'title':'Rise Of AI in India'}

final_state=workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise Of AI in India', 'outline': '**Blog Title:**  \n**The Rise of AI in India – From Startup Hubs to Smart Cities**\n\n---\n\n## 1. Introduction  \n- **Hook:** Quick stats that illustrate India’s AI momentum (e.g., AI market projected to reach $4.5\u202fbn by 2025; 70% of Indian startups using AI).  \n- **Purpose of the Blog:** Explain why AI is a game‑changer for India, what the blog will cover, and who will benefit (entrepreneurs, policy makers, students, investors).  \n- **Thesis Statement:** India’s AI ecosystem is rapidly evolving through a confluence of government support, talent, data, and a vibrant startup culture, positioning the country as a global AI hub.\n\n---\n\n## 2. Historical Context  \n| Time Period | Key Milestones | Impact |\n|-------------|----------------|--------|\n| 2000‑2005 | Early AI research at IITs & IIMs | Academic foundations |\n| 2006‑2010 | AI labs at NITI Aayog & NASSCOM | First industry‑academia collaborations |\n| 2011‑2015 | Rise of “AI 

In [15]:
print(final_state['outline'])

**Blog Title:**  
**The Rise of AI in India – From Startup Hubs to Smart Cities**

---

## 1. Introduction  
- **Hook:** Quick stats that illustrate India’s AI momentum (e.g., AI market projected to reach $4.5 bn by 2025; 70% of Indian startups using AI).  
- **Purpose of the Blog:** Explain why AI is a game‑changer for India, what the blog will cover, and who will benefit (entrepreneurs, policy makers, students, investors).  
- **Thesis Statement:** India’s AI ecosystem is rapidly evolving through a confluence of government support, talent, data, and a vibrant startup culture, positioning the country as a global AI hub.

---

## 2. Historical Context  
| Time Period | Key Milestones | Impact |
|-------------|----------------|--------|
| 2000‑2005 | Early AI research at IITs & IIMs | Academic foundations |
| 2006‑2010 | AI labs at NITI Aayog & NASSCOM | First industry‑academia collaborations |
| 2011‑2015 | Rise of “AI as a Service” (AWS, Google Cloud) | Accessibility of AI tools |
| 2

In [16]:
print(final_state['content'])

**The Rise of AI in India – From Startup Hubs to Smart Cities**

---

## 
